In [1]:
import pandas as pd 
import numpy as np
import pubchempy as pcp
import networkx as nx
import matplotlib as plt

from fairchem.core.datasets import AseDBDataset
from rdkit import Chem
from torch_geometric.utils import to_networkx

/Users/karaha/miniconda3/envs/fairchem/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/karaha/miniconda3/envs/fairchem/lib/python3.11/site-packages/torchtnt/utils/version.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
W0402 16:14:32.574000 62338 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# Overview

The goal of this section is to load the OPoly26 dataset and explore the data available within, including size, features, etc. 

In [2]:
# given the local dataset path, loads the first .aselmdb file

dataset_path = "../data/train/data0001.aselmdb" 
dataset = AseDBDataset({"src": dataset_path})

In [3]:
# perform object introspection: view attributes and methods available for this dataset 
dir(dataset)

['__abstractmethods__',
 '__add__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__del__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_idlen_cumulative',
 '_is_protocol',
 '_load_dataset_get_ids',
 '_metadata',
 'a2g',
 'atoms_transform',
 'config',
 'connect_db',
 'db_ids',
 'dbs',
 'get_atoms',
 'get_metadata',
 'get_relaxed_energy',
 'ids',
 'indices',
 'key_mapping',
 'lin_ref',
 'metadata_hasattr',
 'num_samples',
 'paths',
 'sample_property_metadata',
 'select_args',
 'transforms']

This dataset contains a variety of useful attributes and methods that allow users to easily access information about the compounds in this dataset. Below, we explore some of these attributes and methods to get a better understanding of the data and functions: 

In [4]:
# allows users to easily get information about the number of compounds in this dataset 
num_samples = dataset.num_samples
print(f"Number of samples in dataset: {num_samples}")

Number of samples in dataset: 73786


In [ ]:
# returns the ids of all compounds in the dataset as a list
ids = dataset.ids

print(f"ids: {ids[:10]}") # print with truncated output; 7000 entries 
print(type(ids))

ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
<class 'list'>


In [6]:
# using index, allows users to access compounds by id/index
# below, we retrieve the first compound in the dataset 
atom0 = dataset.get_atoms(0)

print(atom0)
print(type(atom0))

Atoms(symbols='C86H119N17O16', pbc=False, calculator=SinglePointCalculator(...))
<class 'ase.atoms.Atoms'>


Compounds are stored in this dataset as Atoms objects. Calling `get_atoms()` creates an instance of the Atoms class, which contains all the information for the specified compound. In a later section, we will further explore the attributes and methods available within each class. We will continue with the main dataset for now.

The `.a2g` method for this dataset is interesting. It is an instance of the `AtomsToGraphs` class, with some pre-defined parameters. We will explore this further, as the ability to represent molecules as graph objects will be useful for machine learning applications. 

In [7]:
help(dataset.a2g)

Help on AtomsToGraphs in module fairchem.core.preprocessing.atoms_to_graphs object:

class AtomsToGraphs(builtins.object)
 |  AtomsToGraphs(max_neigh: 'int' = 200, radius: 'int' = 6, r_energy: 'bool' = False, r_forces: 'bool' = False, r_distances: 'bool' = False, r_edges: 'bool' = True, r_fixed: 'bool' = True, r_pbc: 'bool' = False, r_stress: 'bool' = False, r_data_keys: 'Sequence[str] | None' = None, molecule_cell_size: 'float | None' = None) -> 'None'
 |  
 |  A class to help convert periodic atomic structures to graphs.
 |  
 |  The AtomsToGraphs class takes in periodic atomic structures in form of ASE atoms objects and converts
 |  them into graph representations for use in PyTorch. The primary purpose of this class is to determine the
 |  nearest neighbors within some radius around each individual atom, taking into account PBC, and set the
 |  pair index and distance between atom pairs appropriately. Lastly, atomic properties and the graph information
 |  are put into a PyTorch ge

In [9]:
# AtomsToGraphs: converts periodic atomic structures to graphs
# input: ASE atoms objects 
# return: list of torch geometric data objects w/ mol graph info -> use with pytorch 

a2g = dataset.a2g # initialize instance of AtomsToGraph converter

# to access pre-defined attributes for this class: 
print(f"Max Neighbors: {a2g.max_neigh}") # dir(a2g) for more attributes

# data = a2g.convert(dataset.get_atoms(0))
# print(data.edge_stores[0])

# G = to_networkx(data, to_undirected=True)

# plt.figure(figsize=(8, 8))
# pos = nx.spring_layout(G)  
# nx.draw(G, pos, with_labels=True)
# plt.title("atom0")
# plt.show()

Max Neighbors: 200
